# MLFlow Tracking Service

In [21]:
import mlflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from mlflow.models import infer_signature
from mlflow import MlflowClient


In [22]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Hyperparameter Search")

<Experiment: artifact_location='file:///C:/Users/swkra/OneDrive/Υπολογιστής/mlops/mlartifacts/1', creation_time=1785278305109, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785278305109, lifecycle_stage='active', name='Hyperparameter Search', tags={}, trace_location=None, workspace='default'>

In [23]:
## load the data
X,y = datasets.load_iris(return_X_y=True)

## split the data into train and test
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2)


In [24]:
best_accuracy = -1
best_run_id = None

In [25]:
best_accuracy = -1
best_model_info = None

for C in [0.01, 0.1, 1, 10]:

    with mlflow.start_run():

        model = LogisticRegression(
            C=C,
            max_iter=1000,
            random_state=42
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)

        mlflow.log_param("C", C)
        mlflow.log_metric("accuracy", accuracy)

        model_name = f"logreg_C_{str(C).replace('.', '_')}"

        model_info = mlflow.sklearn.log_model(
            model,
            name=model_name
        )

        if accuracy > best_accuracy:

            best_accuracy = accuracy
            best_model_info = model_info

print(best_accuracy)

2026/07/29 01:53:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run skillful-shrike-825 at: http://127.0.0.1:5000/#/experiments/1/runs/ec7dcf10801247db9683f68e53d54809
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/07/29 01:53:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run lyrical-mule-775 at: http://127.0.0.1:5000/#/experiments/1/runs/61d1f21cdc744ff8bf8e1809d65b023f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/07/29 01:53:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run learned-asp-174 at: http://127.0.0.1:5000/#/experiments/1/runs/0bad64e92d904e109548691d8e7dce7f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/07/29 01:53:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run delicate-zebra-699 at: http://127.0.0.1:5000/#/experiments/1/runs/e3c9c2610cb146f0b92f609c9329bf8c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
0.9333333333333333


In [26]:
mlflow.register_model(
    best_model_info.model_uri,
    "iris_classifier"
)

Registered model 'iris_classifier' already exists. Creating a new version of this model...
2026/07/29 01:53:51 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris_classifier, version 3
Created version '3' of model 'iris_classifier'.


<ModelVersion: aliases=[], creation_timestamp=1785279230974, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1785279230974, metrics=None, model_id=None, name='iris_classifier', params=None, run_id='e3c9c2610cb146f0b92f609c9329bf8c', run_link='', source='models:/m-6aa0b58d6b2648e78b7bc0d918800cb4', status='READY', status_message=None, tags={}, user_id='', version='3', workspace='default'>

In [27]:
client = MlflowClient()

current = client.get_model_version(
    "iris_classifier",
    "1"
)

current_run = client.get_run(current.run_id)

current_accuracy = float(
    current_run.data.metrics["accuracy"]
)

In [28]:
best_new_accuracy = -1
best_new_model = None

solver = "saga"

for C in [0.5, 1, 5, 20]:

    with mlflow.start_run():

        mlflow.set_tag("mlflow.runName", f"C={C}, solver={solver}")

        model = LogisticRegression(
            C=C,
            solver=solver,
            max_iter=2000,
            random_state=42
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)

        mlflow.log_param("C", C)
        mlflow.log_param("solver", solver)
        mlflow.log_metric("accuracy", accuracy)

        model_name = (
            f"logreg_C_{str(C).replace('.', '_')}"
            f"_solver_{solver}"
        )

        model_info = mlflow.sklearn.log_model(
            model,
            name=model_name
        )

        if accuracy > best_new_accuracy:
            best_new_accuracy = accuracy
            best_new_model = model_info

print(f"Best new accuracy: {best_new_accuracy:.4f}")

2026/07/29 01:53:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run C=0.5, solver=saga at: http://127.0.0.1:5000/#/experiments/1/runs/bb06ba7d771c493998923ea60478f4d0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/07/29 01:53:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run C=1, solver=saga at: http://127.0.0.1:5000/#/experiments/1/runs/8988195fe4f74176b83043dc4c8ecf7f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/07/29 01:54:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run C=5, solver=saga at: http://127.0.0.1:5000/#/experiments/1/runs/5069733736ed4f768752032687b1dabf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/07/29 01:54:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run C=20, solver=saga at: http://127.0.0.1:5000/#/experiments/1/runs/3df2419f0bc349beac04b15c56598d2a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Best new accuracy: 0.9333


In [29]:
print("Registered model:", current_accuracy)
print("Best new model:", best_new_accuracy)

if best_new_accuracy > current_accuracy:

    print("New best model found!")

    mlflow.register_model(
        best_new_model.model_uri,
        "iris_classifier"
    )

else:

    print("Current registered model remains the champion.")

Registered model: 0.9666666666666667
Best new model: 0.9333333333333333
Current registered model remains the champion.
